# Persistencia y consultas avanzadas

Este notebook deja lista la base `ecommerce`: conexión, schema, carga con `COPY` y validación. Al final queda el método `consultar()` para trabajar las preguntas de negocio.

## 1. Conexión

In [2]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "db" / "connection.py").exists():
    ROOT = Path(r"C:\Programacion\DataScienceIA\Tutor_Dev_Senior_Code\Tutorias_Cohorte_6\Tutorias_Refuerzo_IA6\Tutoria_persistencia_consultas_avanzadas")

sys.path.insert(0, str(ROOT / "db"))

from connection import get_connection  # pyright: ignore[reportMissingImports]
from load_data import (
    COPY_ORDER,
    EXPECTED_COUNTS,
    SCHEMA_PATH,
    apply_schema,
    copy_csv,
    validate_counts,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("ROOT:", ROOT)

ROOT: C:\Programacion\DataScienceIA\Tutor_Dev_Senior_Code\Tutorias_Cohorte_6\Tutorias_Refuerzo_IA6\Tutoria_persistencia_consultas_avanzadas


In [3]:
conn = get_connection()
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            current_database() AS base_datos,
            current_user AS usuario,
            inet_server_addr() AS host,
            inet_server_port() AS puerto,
            version() AS version_postgresql
        """
    )
    info = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])

print("Conexion verificada")
info

Conexion verificada


,base_datos,usuario,host,puerto,version_postgresql
0,ecommerce,postgres,::1,5432,"PostgreSQL 17.4 on x86_64-windows, compiled by..."


## 2. Schema

El DDL vive en `db/schema.sql`. Incluye PK, UNIQUE, NOT NULL, CHECK y FK. Aplicarlo recrea las tablas.

In [4]:
schema_sql = SCHEMA_PATH.read_text(encoding="utf-8")
print(schema_sql)

DROP TABLE IF EXISTS ventas CASCADE;
DROP TABLE IF EXISTS productos CASCADE;
DROP TABLE IF EXISTS clientes CASCADE;

CREATE TABLE clientes (
    cliente_id      INTEGER         PRIMARY KEY,
    nombre          VARCHAR(100)    NOT NULL,
    email           VARCHAR(150)    NOT NULL UNIQUE,
    ciudad          VARCHAR(100)    NOT NULL,
    pais            VARCHAR(50)     NOT NULL,
    segmento        VARCHAR(20)     NOT NULL,
    fecha_registro  DATE            NOT NULL,
    CONSTRAINT ck_clientes_nombre_no_vacio
        CHECK (btrim(nombre) <> ''),
    CONSTRAINT ck_clientes_email_formato
        CHECK (email ~* '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'),
    CONSTRAINT ck_clientes_pais
        CHECK (pais IN ('Argentina', 'Chile', 'Colombia', 'Ecuador', 'Espana', 'Mexico', 'Peru')),
    CONSTRAINT ck_clientes_segmento
        CHECK (segmento IN ('bronce', 'plata', 'oro', 'platino'))
);

CREATE TABLE productos (
    producto_id     INTEGER         PRIMARY KEY,
    nombre        

## 3. Carga con COPY

Orden obligatorio: `clientes` → `productos` → `ventas`.

In [5]:
def preparar_base(forzar=False):
    """Aplica schema, carga los CSV y deja la base lista para consultar."""
    with conn.cursor() as cur:
        ya_ok = False
        if not forzar:
            try:
                counts = validate_counts(cur)
                ya_ok = all(ok for _, _, ok in counts.values())
            except Exception:
                ya_ok = False

        if ya_ok:
            print("Datos ya cargados. Se omite recrear tablas.")
            print("Usa preparar_base(forzar=True) si quieres volver a cargar.")
            return

        print("Aplicando schema...")
        apply_schema(cur)

        for table_name in COPY_ORDER:
            print(f"COPY {table_name}...")
            copy_csv(cur, table_name)

        print("Carga completada.")


preparar_base(forzar=False)

Datos ya cargados. Se omite recrear tablas.
Usa preparar_base(forzar=True) si quieres volver a cargar.


## 4. Validación de la carga

In [6]:
def validar_carga():
    checks = []

    with conn.cursor() as cur:
        counts = validate_counts(cur)
        for tabla, (actual, esperado, ok) in counts.items():
            checks.append(
                {
                    "chequeo": f"COUNT(*) {tabla}",
                    "detalle": f"{actual} (esperado {esperado})",
                    "ok": ok,
                }
            )

        cur.execute(
            """
            SELECT 'clientes' AS tabla, COUNT(*) AS n, COUNT(DISTINCT cliente_id) AS n_pk FROM clientes
            UNION ALL
            SELECT 'productos', COUNT(*), COUNT(DISTINCT producto_id) FROM productos
            UNION ALL
            SELECT 'ventas', COUNT(*), COUNT(DISTINCT venta_id) FROM ventas
            """
        )
        for tabla, n, n_pk in cur.fetchall():
            checks.append(
                {
                    "chequeo": f"PK unica {tabla}",
                    "detalle": f"filas={n}, distinct_pk={n_pk}",
                    "ok": n == n_pk,
                }
            )

        cur.execute(
            """
            SELECT
                COUNT(*) FILTER (WHERE cliente_id IS NULL) AS cliente_id_nulos,
                COUNT(*) FILTER (WHERE nombre IS NULL) AS nombre_nulos,
                COUNT(*) FILTER (WHERE email IS NULL) AS email_nulos,
                COUNT(*) - COUNT(DISTINCT email) AS emails_duplicados
            FROM clientes
            """
        )
        cliente_id_nulos, nombre_nulos, email_nulos, emails_duplicados = cur.fetchone()
        checks.append({"chequeo": "clientes sin nulos PK/nombre/email", "detalle": f"nulls=({cliente_id_nulos},{nombre_nulos},{email_nulos})", "ok": cliente_id_nulos == nombre_nulos == email_nulos == 0})
        checks.append({"chequeo": "emails unicos", "detalle": f"duplicados={emails_duplicados}", "ok": emails_duplicados == 0})

        cur.execute(
            """
            SELECT
                COUNT(*) FILTER (WHERE producto_id IS NULL) AS producto_id_nulos,
                COUNT(*) FILTER (WHERE nombre IS NULL) AS nombre_nulos,
                COUNT(*) - COUNT(DISTINCT nombre) AS nombres_duplicados
            FROM productos
            """
        )
        producto_id_nulos, nombre_nulos, nombres_duplicados = cur.fetchone()
        checks.append({"chequeo": "productos sin nulos PK/nombre", "detalle": f"nulls=({producto_id_nulos},{nombre_nulos})", "ok": producto_id_nulos == nombre_nulos == 0})
        checks.append({"chequeo": "nombres de producto unicos", "detalle": f"duplicados={nombres_duplicados}", "ok": nombres_duplicados == 0})

        cur.execute(
            """
            SELECT COUNT(*) FILTER (
                WHERE venta_id IS NULL
                   OR cliente_id IS NULL
                   OR producto_id IS NULL
                   OR fecha_venta IS NULL
                   OR cantidad IS NULL
                   OR total IS NULL
            )
            FROM ventas
            """
        )
        ventas_nulos = cur.fetchone()[0]
        checks.append({"chequeo": "ventas sin nulos en columnas clave", "detalle": f"nulos={ventas_nulos}", "ok": ventas_nulos == 0})

        cur.execute(
            """
            SELECT COUNT(*)
            FROM ventas v
            LEFT JOIN clientes c ON c.cliente_id = v.cliente_id
            WHERE c.cliente_id IS NULL
            """
        )
        huerfanos_cliente = cur.fetchone()[0]
        checks.append({"chequeo": "FK ventas -> clientes", "detalle": f"huerfanos={huerfanos_cliente}", "ok": huerfanos_cliente == 0})

        cur.execute(
            """
            SELECT COUNT(*)
            FROM ventas v
            LEFT JOIN productos p ON p.producto_id = v.producto_id
            WHERE p.producto_id IS NULL
            """
        )
        huerfanos_producto = cur.fetchone()[0]
        checks.append({"chequeo": "FK ventas -> productos", "detalle": f"huerfanos={huerfanos_producto}", "ok": huerfanos_producto == 0})

        cur.execute(
            """
            SELECT conrelid::regclass::text AS tabla, contype, conname
            FROM pg_constraint
            WHERE connamespace = 'public'::regnamespace
            ORDER BY tabla, contype, conname
            """
        )
        restricciones = pd.DataFrame(cur.fetchall(), columns=["tabla", "tipo", "nombre"])

    resultado = pd.DataFrame(checks)
    resultado["estado"] = resultado["ok"].map({True: "OK", False: "ERROR"})
    if not resultado["ok"].all():
        raise ValueError("La validacion de la carga fallo. Revisa la tabla de chequeos.")

    print("Validacion de carga: OK")
    return resultado.drop(columns=["ok"]), restricciones


resumen_validacion, restricciones = validar_carga()
resumen_validacion

Validacion de carga: OK


,chequeo,detalle,estado
0,COUNT(*) clientes,10000 (esperado 10000),OK
1,COUNT(*) productos,2000 (esperado 2000),OK
2,COUNT(*) ventas,200000 (esperado 200000),OK
3,PK unica clientes,"filas=10000, distinct_pk=10000",OK
4,PK unica productos,"filas=2000, distinct_pk=2000",OK
5,PK unica ventas,"filas=200000, distinct_pk=200000",OK
6,clientes sin nulos PK/nombre/email,"nulls=(0,0,0)",OK
7,emails unicos,duplicados=0,OK
8,productos sin nulos PK/nombre,"nulls=(0,0)",OK
9,nombres de producto unicos,duplicados=0,OK


In [7]:
print("Restricciones aplicadas")
restricciones

Restricciones aplicadas


,tabla,tipo,nombre
0,clientes,c,ck_clientes_email_formato
1,clientes,c,ck_clientes_nombre_no_vacio
2,clientes,c,ck_clientes_pais
3,clientes,c,ck_clientes_segmento
4,clientes,p,clientes_pkey
5,clientes,u,clientes_email_key
6,productos,c,ck_productos_categoria
7,productos,c,ck_productos_coste
8,productos,c,ck_productos_coste_vs_precio
9,productos,c,ck_productos_nombre_no_vacio


## 5. Método para consultar

Usa `consultar("""...""")` desde aquí hacia abajo. Devuelve un DataFrame.

In [8]:
def consultar(sql, params=None):
    """Ejecuta SQL y devuelve un DataFrame."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        if cur.description is None:
            return None
        columnas = [col[0] for col in cur.description]
        filas = cur.fetchall()
    return pd.DataFrame(filas, columns=columnas)


consultar(
    """
    SELECT
        (SELECT COUNT(*) FROM clientes)  AS clientes,
        (SELECT COUNT(*) FROM productos) AS productos,
        (SELECT COUNT(*) FROM ventas)    AS ventas
    """
)

,clientes,productos,ventas
0,10000,2000,200000


## Consultas de negocio

¿Cuanto vendimos, a cuantos clientes, con que ticket medio y en qué periodo?

In [9]:
consulta_01 = consultar("""
SELECT
    COUNT(*)                     AS n_ventas,
    COUNT(DISTINCT cliente_id)   AS cliente_con_compra,
    COUNT(DISTINCT producto_id)  AS productos_vendidos,
    ROUND(SUM(total), 2)         AS ingresos,
    ROUND(AVG(total), 2)         AS ticket_medio,
    ROUND(SUM(cantidad), 2)      AS unidades,
    MIN(fecha_venta)             AS desde,
    MAX(fecha_venta)             AS hasta
FROM ventas
""")

display(consulta_01)

,n_ventas,cliente_con_compra,productos_vendidos,ingresos,ticket_medio,unidades,desde,hasta
0,200000,10000,2000,84251415.73,421.26,433401.00,2022-01-08,2025-12-30


Mapa gaografico de ingreso

¿Qué paises y ciudades concentran el negocio?

In [10]:
consulta_02 = consultar("""
SELECT
    c.pais,
    c.ciudad,
    COUNT(DISTINCT c.cliente_id)  AS n_cliente,
    COUNT(*)                      AS n_ventas,
    ROUND(SUM(v.total), 2)        AS ingresos,
    ROUND(SUM(v.total) / COUNT(DISTINCT c.cliente_id), 2) AS ingresos_por_cliente
FROM ventas v
JOIN clientes c ON c.cliente_id = v.cliente_id
GROUP BY c.pais, c.ciudad
ORDER BY ingresos DESC
    """)

display(consulta_02)

,pais,ciudad,n_cliente,n_ventas,ingresos,ingresos_por_cliente
0,Espana,Valencia,873,17730,7483512.65,8572.18
1,Ecuador,Quito,857,17200,7202909.84,8404.80
2,Mexico,Guadalajara,853,17325,7169397.58,8404.92
3,Chile,Santiago,849,16617,7147517.38,8418.75
4,Peru,Lima,846,16847,7071690.96,8358.97
5,Espana,Barcelona,825,16464,7014592.45,8502.54
6,Colombia,Bogota,820,16502,6974968.59,8506.06
7,Colombia,Cali,826,16857,6962009.08,8428.58
8,Espana,Madrid,788,15860,6911944.57,8771.50
9,Mexico,Ciudad de Mexico,825,16342,6900454.51,8364.19


# Vistas

In [12]:
#Crear una vista del ticket medio que sea enriquecido

#Pregunta: ¿Cómo dejo listo el detalle venta-cliente para no armar el JOIN en cada pregunta?


consultar("""
CREATE OR REPLACE VIEW v_ventas_enriquecidas AS
SELECT
    v.venta_id,
    v.fecha_venta,
    v.cantidad,
    v.total,
    v.canal,
    c.cliente_id,
    c.nombre           AS cliente,
    c.pais,
    c.ciudad,
    c.segmento,
    p.producto_id,
    p.nombre           AS producto,
    p.categoria
FROM ventas v
JOIN clientes c ON c.cliente_id = v.cliente_id
JOIN productos p ON p.producto_id = v.producto_id
          """)

print("Vista creada: v_ventas_enriquecidas")

Vista creada: v_ventas_enriquecidas


A partir de aquí, se consulta como si fuese una tabla. La diferencia: No hay datos copiados, solo el plan de consulta

In [15]:
consultar("""
SELECT
    pais,
    ciudad,
    COUNT(DISTINCT cliente_id)    AS n_cliente,
    COUNT(*)                      AS n_ventas,
    ROUND(SUM(total), 2)          AS ingresos
FROM v_ventas_enriquecidas
GROUP BY pais, ciudad
ORDER BY ingresos DESC
    """)

,pais,ciudad,n_cliente,n_ventas,ingresos
0,Espana,Valencia,873,17730,7483512.65
1,Ecuador,Quito,857,17200,7202909.84
2,Mexico,Guadalajara,853,17325,7169397.58
3,Chile,Santiago,849,16617,7147517.38
4,Peru,Lima,846,16847,7071690.96
5,Espana,Barcelona,825,16464,7014592.45
6,Colombia,Bogota,820,16502,6974968.59
7,Colombia,Cali,826,16857,6962009.08
8,Espana,Madrid,788,15860,6911944.57
9,Mexico,Ciudad de Mexico,825,16342,6900454.51


In [20]:
#Cuando una vista puede agregar

consultar("""
CREATE OR REPLACE VIEW v_kpis_mensuales AS
SELECT
    DATE_TRUNC('month', fecha_venta)::date AS mes,
    COUNT(*)                     AS n_ventas,
    COUNT(DISTINCT cliente_id)    AS clientes,
    ROUND(SUM(total), 2)         AS ingresos,
    ROUND(AVG(total), 2)         AS ticket_medio
FROM ventas
GROUP BY DATE_TRUNC('month', fecha_venta)
          """)

In [21]:
consultar("""
SELECT *
FROM v_kpis_mensuales
ORDER BY mes
LIMIT 8
          """)

,mes,n_ventas,clientes,ingresos,ticket_medio
0,2022-01-01,50,40,18335.62,366.71
1,2022-02-01,141,111,53336.90,378.28
2,2022-03-01,292,219,173263.34,593.37
3,2022-04-01,393,316,182477.22,464.32
4,2022-05-01,548,412,223247.66,407.39
5,2022-06-01,620,472,262780.40,423.84
6,2022-07-01,787,621,340535.57,432.70
7,2022-08-01,959,722,372767.39,388.70


In [24]:
consultar("""
SELECT table_name AS vista
FROM information_schema.views
WHERE table_schema = 'public'
ORDER BY 1
          """)

,vista
0,v_kpis_mensuales
1,v_top_productos_categoria
2,v_ventas_enriquecidas


In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="ecommerce",
    user="postgres",
    password="#####"
)

df = pd.read_sql("""
SELECT
    DATE_TRUNC('month', fecha_venta)::date AS mes,
    COUNT(*)                     AS n_ventas,
    COUNT(DISTINCT cliente_id)    AS clientes,
    ROUND(SUM(total), 2)         AS ingresos,
    ROUND(AVG(total), 2)         AS ticket_medio
FROM ventas
GROUP BY DATE_TRUNC('month', fecha_venta)
                 """, conn)


C:\Users\JORGE\AppData\Local\Temp\ipykernel_14056\3664748928.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


In [26]:
df

,mes,n_ventas,clientes,ingresos,ticket_medio
0,2022-01-01,50,40,18335.62,366.71
1,2022-02-01,141,111,53336.90,378.28
2,2022-03-01,292,219,173263.34,593.37
3,2022-04-01,393,316,182477.22,464.32
4,2022-05-01,548,412,223247.66,407.39
5,2022-06-01,620,472,262780.40,423.84
6,2022-07-01,787,621,340535.57,432.70
7,2022-08-01,959,722,372767.39,388.70
8,2022-09-01,1020,763,426419.56,418.06
9,2022-10-01,1217,900,560087.42,460.22
